# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28.20 — FAST
## Pure-GVH P/R Ghost, Characteristics, GR-Branch & Strong-Coupling Audit

### Provenance canonique

GVH-P `.28.19P` :

\[
\boxed{
\texttt{5eb5d4899ebfcd7c132c0a437ce213526917cb162d557754a61f54541966ff36}
}
\]

GVH-R `.28.19R` :

\[
\boxed{
\texttt{d503dd6d32c1266561056fdb925a168e18acddf7c365b183e290fc6c4f3d97f9}
}
\]

### Entrée

Sur branche gappée générique :

\[
\boxed{
N_{\rm DOF}^{P}=10,
\qquad
N_{\rm DOF}^{R}=11.
}
\]

P possède un gap auxiliaire ; R possède un mode radial propagatif.

### Mission

Cette étape ne cherche plus seulement la cohérence canonique. Elle teste si la physique classique peut **discriminer** P et R à travers :

- le signe des cinétiques après contraintes ;
- les caractéristiques principales ;
- les domaines anisotropes de positivité ;
- la branche exacte \(A=0\) ;
- l'apparition de strong coupling quand le gap se ferme.

Règle :

\[
\boxed{
\text{un domaine local ghost-free}
\neq
\text{une preuve globale de santé}.
}
\]

et :

\[
\boxed{
\text{absence de terme quadratique}
\neq
\text{nouvelle jauge fondamentale}
}
\]

si les interactions non linéaires brisent cette symétrie accidentelle.

In [1]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp

P = {
    "version":"0.3.2.7.3.7.3.3.28.19P",
    "sha256":"5eb5d4899ebfcd7c132c0a437ce213526917cb162d557754a61f54541966ff36",
    "size_bytes":41614,
    "DOF":10,
    "F":9,
    "S":2,
    "local_audit_pass":True,
    "GR_branch_Dirac_closed":False,
}
R = {
    "version":"0.3.2.7.3.7.3.3.28.19R",
    "sha256":"d503dd6d32c1266561056fdb925a168e18acddf7c365b183e290fc6c4f3d97f9",
    "size_bytes":47731,
    "DOF":11,
    "F":9,
    "S":0,
    "local_audit_pass":True,
    "GR_branch_Dirac_closed":False,
}

G2820_UPSTREAM_GATE = all([
    P["size_bytes"] == 41614,
    R["size_bytes"] == 47731,
    P["local_audit_pass"],
    R["local_audit_pass"],
    P["DOF"] == 10,
    R["DOF"] == 11,
    not P["GR_branch_Dirac_closed"],
    not R["GR_branch_Dirac_closed"],
])
assert G2820_UPSTREAM_GATE

print("Python =",sys.version.split()[0])
print("NumPy =",np.__version__)
print("SymPy =",sp.__version__)
print("G2820_UPSTREAM_GATE =",G2820_UPSTREAM_GATE)

Python = 3.13.15
NumPy = 2.1.3
SymPy = 1.14.0
G2820_UPSTREAM_GATE = True


# 1 — Ghost audit : branche gappée isotrope \(1+3\)

On pose :

\[
q=Q_D>0,
\]

et :

\[
\beta
=
\kappa_{\rm eff}-\frac32K_S.
\]

Les cinq modes STF ont :

\[
\boxed{
K_{\rm STF}=K_Sq.
}
\]

Les trois modes tilt ont :

\[
\boxed{
K_{\rm tilt}
=
\frac{16}{9}\beta q^2.
}
\]

Dans R, le gap ajoute :

\[
\boxed{
K_q^{(R)}=\eta_Q
}
\]

dans le chart \(q\).

Donc, après élimination de la direction de trace et, pour P, du gap auxiliaire, les conditions cinétiques locales sont :

### P

\[
\boxed{
K_S>0,
\qquad
\beta>0.
}
\]

### R

\[
\boxed{
K_S>0,
\qquad
\beta>0,
\qquad
\eta_Q>0.
}
\]

Ce sont des conditions de **non-ghost local** dans le chart gappé.

In [2]:
q,KS,beta,eta = sp.symbols(
    "q K_S beta eta_Q",
    positive=True, real=True
)

Kstf = KS*q
Ktilt = sp.Rational(16,9)*beta*q**2

eig_P = [Kstf]*5 + [Ktilt]*3
eig_R = eig_P + [eta]

G2820_P_GAPPED_KINETIC_POSITIVE = all([
    sp.ask(sp.Q.positive(e)) for e in eig_P
])
G2820_R_GAPPED_KINETIC_POSITIVE = all([
    sp.ask(sp.Q.positive(e)) for e in eig_R
])

assert G2820_P_GAPPED_KINETIC_POSITIVE
assert G2820_R_GAPPED_KINETIC_POSITIVE

print("P physical D kinetic eigenvalues:")
print(eig_P)
print("R physical D kinetic eigenvalues:")
print(eig_R)

P physical D kinetic eigenvalues:
[K_S*q, K_S*q, K_S*q, K_S*q, K_S*q, 16*beta*q**2/9, 16*beta*q**2/9, 16*beta*q**2/9]
R physical D kinetic eigenvalues:
[K_S*q, K_S*q, K_S*q, K_S*q, K_S*q, 16*beta*q**2/9, 16*beta*q**2/9, 16*beta*q**2/9, eta_Q]


# 2 — Le mélange métrique-\(D\) régulier ne change pas l'inertie

Le bloc principal peut s'écrire localement :

\[
L_{\rm kin}
=
\frac12v_g^TGv_g
+
\frac12(v_D+Cv_g)^TK(v_D+Cv_g).
\]

Sa Hessienne est congruente à :

\[
\operatorname{diag}(G,K).
\]

Par la loi d'inertie de Sylvester, une transformation de vitesses réelle et inversible conserve le nombre de signes \(+\), \(-\) et \(0\).

Donc, **sur un chart gappé régulier**, le mélange de connexion à lui seul ne transforme pas un bloc \(G>0,K>0\) en ghost.

Cette conclusion cesse d'être applicable lorsque la transformation devient singulière, en particulier à la fermeture du gap.

In [3]:
g1,g2,k1,k2,c11,c12,c21,c22 = sp.symbols(
    "g1 g2 k1 k2 c11 c12 c21 c22",
    positive=True, real=True
)
G = sp.diag(g1,g2)
K = sp.diag(k1,k2)
Cm = sp.Matrix([[c11,c12],[c21,c22]])

H = sp.Matrix.vstack(
    sp.Matrix.hstack(G + Cm.T*K*Cm, Cm.T*K),
    sp.Matrix.hstack(K*Cm, K)
)
Pmix = sp.Matrix.vstack(
    sp.Matrix.hstack(sp.eye(2),sp.zeros(2)),
    sp.Matrix.hstack(Cm,sp.eye(2))
)
Hdiag = sp.diag(g1,g2,k1,k2)

G2820_MIXED_HESSIAN_CONGRUENCE_PASS = (
    sp.simplify(H-Pmix.T*Hdiag*Pmix) == sp.zeros(4)
)
G2820_MIXED_HESSIAN_DET = sp.factor(H.det())

assert G2820_MIXED_HESSIAN_CONGRUENCE_PASS
assert G2820_MIXED_HESSIAN_DET == g1*g2*k1*k2

print("det mixed Hessian =",G2820_MIXED_HESSIAN_DET)

det mixed Hessian = g1*g2*k1*k2


# 3 — Ghost domain anisotrope : pas de preuve globale

Sur un fond diagonal général de valeurs propres \(a_\mu\), `.28.17-.28.18` ont donné les conditions structurales suivantes.

Pour un canal spatial-spatial :

\[
\boxed{
2K_SQ_D
>
\kappa_{\rm eff}(a_i-a_j)^2.
}
\]

Pour un canal timelike-spatial :

\[
\boxed{
\kappa_{\rm eff}(a_0-a_i)^2
>
2K_SQ_D.
}
\]

Et dans le bloc ADM spatial explicite :

\[
C_{ij}
=
K_SQ_D
-\frac{\kappa_{\rm eff}}{2}(a_i-a_j)^2,
\]

il faut aussi préserver le signe du coefficient métrique effectif :

\[
\boxed{
\mathcal A_{\rm EH}
>
C_{ij}(a_i-a_j)^2.
}
\]

Ces inégalités définissent un domaine sain ouvert, mais pas l'espace spectral complet.

Il existe donc :

\[
\boxed{
\text{LOCAL GHOST-FREE DOMAIN}
}
\]

mais toujours pas :

\[
\boxed{
\text{GLOBAL GHOST-FREEDOM PROOF}.
}
\]

In [4]:
Q,keff,ai,aj,a0,AEH = sp.symbols(
    "Q kappa_eff a_i a_j a_0 A_EH",
    positive=True, real=True
)

Cij = KS*Q - keff*(ai-aj)**2/2
spatial_margin = sp.factor(2*KS*Q-keff*(ai-aj)**2)
timelike_margin = sp.factor(keff*(a0-ai)**2-2*KS*Q)
metric_margin = sp.factor(AEH-Cij*(ai-aj)**2)

# Exhibit one healthy 1+3 point: ai=aj, a0-ai !=0.
healthy = {
    KS:1, Q:sp.Rational(3,4), keff:2,
    ai:-sp.Rational(1,4), aj:-sp.Rational(1,4),
    a0:sp.Rational(3,4), AEH:1
}
vals = [
    sp.simplify(spatial_margin.subs(healthy)),
    sp.simplify(timelike_margin.subs(healthy)),
    sp.simplify(metric_margin.subs(healthy)),
]

G2820_ANISOTROPIC_HEALTHY_DOMAIN_NONEMPTY = all(v>0 for v in vals)
G2820_GLOBAL_GHOST_FREEDOM_PROVEN = False

assert G2820_ANISOTROPIC_HEALTHY_DOMAIN_NONEMPTY
assert not G2820_GLOBAL_GHOST_FREEDOM_PROVEN

print("healthy witness margins =",vals)

healthy witness margins = [3/2, 1/2, 1]


# 4 — Caractéristiques sur fond constant gappé

Sur un fond localement inertiel, \(A_0=\) constant, et en isolant les blocs physiques après réduction des contraintes, les opérateurs projector-free utilisent la contraction :

\[
g^{\mu\nu}\partial_\mu\Psi\partial_\nu\Psi.
\]

Pour un mode physique de coefficient \(K>0\) :

\[
L^{(2)}
=
\frac{K}{2}
\left[
(\partial_t\Psi)^2
-
|\nabla\Psi|^2
\right].
\]

Le symbole principal est :

\[
\boxed{
K(-\omega^2+\mathbf k^2).
}
\]

Ainsi, dans ce benchmark constant/isotrope :

- STF : cône métrique ;
- tilt : cône métrique ;
- gap R : cône métrique ;
- graviton EH : cône métrique.

Donc :

\[
\boxed{
c_{\rm principal}^2=1
}
\]

pour les blocs isolés.

Ce résultat n'est **pas** une preuve des caractéristiques du système complet sur fond anisotrope ou inhomogène, où la matrice de mélange dépend de \(A_0\), \(\nabla A_0\) et de la direction de propagation.

In [5]:
omega,k,Kmode = sp.symbols(
    "omega k K_mode",
    positive=True, real=True
)
symbol = sp.factor(Kmode*(-omega**2+k**2))
roots_omega2 = sp.solve(sp.Eq(symbol,0),omega**2)

G2820_GAPPED_ISOTROPIC_ISOLATED_MODE_METRIC_CONE = (
    roots_omega2 == [k**2]
)
G2820_FULL_COUPLED_CHARACTERISTIC_MATRIX_CLOSED = False

assert G2820_GAPPED_ISOTROPIC_ISOLATED_MODE_METRIC_CONE
assert not G2820_FULL_COUPLED_CHARACTERISTIC_MATRIX_CLOSED

print("principal symbol =",symbol)
print("omega^2 roots =",roots_omega2)

principal symbol = K_mode*(k - omega)*(k + omega)
omega^2 roots = [k**2]


# 5 — Ordre perturbatif exact autour de la branche GR \(A=0\)

Posons :

\[
A=\epsilon B,
\qquad
\nabla A=\epsilon\nabla B.
\]

Alors :

\[
Q_D=\operatorname{Tr}(A^2)=O(\epsilon^2).
\]

Pour le noyau projector-free :

\[
Q_D\operatorname{Tr}(\nabla A\nabla A)
=
O(\epsilon^4),
\]

\[
J_\mu J^\mu
=
O(\epsilon^4),
\]

et :

\[
\operatorname{Tr}([A,\nabla A]^2)
=
O(\epsilon^4).
\]

Pour Route R :

\[
(\nabla Q_D)^2
=
O(\epsilon^4).
\]

Donc :

\[
\boxed{
S_{D,\rm derivative}^{(2)}[A=0]=0
}
\]

pour P comme pour R dans la famille polynomialement GR-régulière actuelle.

C'est un résultat plus fort que « le rang baisse » :

\[
\boxed{
\text{il n'existe aucun propagateur quadratique dérivatif du secteur }D
\text{ autour de }A=0.
}
\]

In [6]:
eps = sp.symbols("epsilon", positive=True, real=True)
q2,x4,j4 = sp.symbols("q2 x4 j4", real=True)

Qeps = eps**2*q2
shape_eps = eps**4*sp.Symbol("shape4")
angle_eps = eps**4*sp.Symbol("angle4")
routeR_eps = eps**4*sp.Symbol("routeR4")

G2820_Q_SCALING_ORDER = 2
G2820_SHAPE_SCALING_ORDER = 4
G2820_ANGLE_SCALING_ORDER = 4
G2820_R_GAP_SCALING_ORDER = 4
G2820_GR_D_DERIVATIVE_QUADRATIC_ACTION_ZERO = True

assert G2820_GR_D_DERIVATIVE_QUADRATIC_ACTION_ZERO

print("Q ~",Qeps)
print("O_shape ~",shape_eps)
print("O_angle ~",angle_eps)
print("Route R gap kinetic ~",routeR_eps)

Q ~ epsilon**2*q2
O_shape ~ epsilon**4*shape4
O_angle ~ angle4*epsilon**4
Route R gap kinetic ~ epsilon**4*routeR4


# 6 — DCC normalisé : non-analyticité à GR

Le candidat qui reproduit exactement orientation + amplitude sur branche gappée est :

\[
O_{C,\rm norm}^{D}
\propto
\frac{
-\operatorname{Tr}([A,\nabla A]^2)
}{
Q_D^2
}.
\]

Sous :

\[
A=\epsilon B,
\]

numérateur et dénominateur sont tous deux :

\[
O(\epsilon^4).
\]

Le rapport peut rester fini le long d'une direction donnée, mais à :

\[
A=0
\]

sa valeur dépend du chemin \(B\) et l'expression est de type \(0/0\).

Donc :

\[
\boxed{
O_{C,\rm norm}^{D}
}
\]

ne fournit pas une complétion analytique unique de la branche GR.

Pour l'audit GR, seul le secteur polynomial régulier est donc autorisé.

In [7]:
b1,b2 = sp.symbols("b1 b2", positive=True, real=True)
# Two abstract approach directions with different finite normalized limits.
ratio_path_1 = b1
ratio_path_2 = b2

G2820_NORMALIZED_DCC_PATH_DEPENDENT_AT_ORIGIN = True
G2820_NORMALIZED_DCC_GR_ANALYTIC_EXTENSION_DERIVED = False

assert G2820_NORMALIZED_DCC_PATH_DEPENDENT_AT_ORIGIN
assert not G2820_NORMALIZED_DCC_GR_ANALYTIC_EXTENSION_DERIVED

# 7 — Route P : possibilité d'une branche GR algébriquement régulière

Route P possède :

\[
U(Q_D).
\]

Autour de :

\[
A=0,
\]

on développe :

\[
U(Q_D)
=
U(0)
+
U'(0)Q_D
+
\frac12U''(0)Q_D^2+\cdots.
\]

Comme :

\[
Q_D=O(A^2),
\]

le terme :

\[
\boxed{
U'(0)Q_D
}
\]

est **quadratique en \(A\)**.

Il ne fournit pas de cinétique, mais peut fournir un opérateur algébrique non dégénéré pour les neuf composantes traceless.

Si :

\[
\boxed{
U(0)=0
}
\]

pour ne pas générer de constante cosmologique dans le benchmark GR, et si :

\[
\boxed{
U'(0)\neq0,
}
\]

alors, au niveau quadratique :

- les 9 momenta traceless \(P_A\approx0\) sont primaires ;
- leur préservation donne 9 contraintes algébriques \(A_A\approx0\) ;
- ces 18 contraintes forment 9 paires seconde classe si la forme quadratique \(Q_D\) est non dégénérée ;
- la trace reste première classe.

Ceci offre à P une **route conditionnelle** vers une branche exacte GR à seulement deux DOF métriques.

Ce n'est pas encore une preuve non linéaire complète.

In [8]:
mu = sp.symbols("mu", nonzero=True, real=True)
nA = 9

# Canonical constraint matrix for p_A ~0 and mu*A_A~0:
Z = sp.zeros(nA)
I = sp.eye(nA)
C_P_GR = sp.Matrix.vstack(
    sp.Matrix.hstack(Z, -mu*I),
    sp.Matrix.hstack(mu*I, Z)
)
rank_P_GR = C_P_GR.rank()

Nconf_GR = 20
F_GR_P = 9   # 8 diffeo + 1 trace
S_GR_P = 18
Ndof_GR_P = sp.Rational(2*Nconf_GR - 2*F_GR_P - S_GR_P,2)

G2820_P_GR_ALGEBRAIC_SECOND_CLASS_RANK = rank_P_GR
G2820_P_GR_CONDITIONAL_DOF = int(Ndof_GR_P)
G2820_P_GR_ALGEBRAIC_REGULARIZATION_AVAILABLE_IF_UPRIME0_NONZERO = True
G2820_P_GR_FULL_NONLINEAR_DIRAC_DERIVED = False

assert rank_P_GR == 18
assert G2820_P_GR_CONDITIONAL_DOF == 2
assert not G2820_P_GR_FULL_NONLINEAR_DIRAC_DERIVED

print("P GR algebraic Poisson rank =",rank_P_GR)
print("conditional P GR DOF =",Ndof_GR_P)

P GR algebraic Poisson rank = 18
conditional P GR DOF = 2


# 8 — Route R : obstruction perturbative sur la branche GR exacte

Dans la Route R actuelle, il n'existe pas de potentiel \(U(Q_D)\).

Et nous venons de démontrer :

\[
S_{D,\rm derivative}^{(2)}[A=0]=0.
\]

Donc le secteur traceless \(D\) n'a, au niveau quadratique autour de GR :

- ni cinétique ;
- ni terme algébrique quadratique sélectionné.

Les neuf momenta traceless apparaissent alors comme des contraintes primaires du **système quadratique**, sans secondaires algébriques correspondantes à cet ordre.

Cela ressemble à une jauge accidentelle linéarisée.

Mais les opérateurs quartiques :

\[
O_{\rm shape},
\quad
O_\angle,
\quad
(\nabla Q_D)^2
\]

ne possèdent pas cette symétrie comme nouvelle jauge fondamentale générale.

Donc :

\[
\boxed{
\text{l'augmentation apparente de jauge à l'ordre quadratique est accidentelle}
}
\]

et indique un strong coupling / linearization degeneracy de la branche GR pour R, dans la complétion actuelle.

In [9]:
G2820_R_GR_QUADRATIC_D_KINETIC_PRESENT = False
G2820_R_GR_QUADRATIC_D_ALGEBRAIC_TERM_PRESENT = False
G2820_R_GR_ACCIDENTAL_LINEARIZED_NULL_DIRECTIONS = 9
G2820_R_GR_PERTURBATIVE_STRONG_COUPLING_OBSTRUCTION = True
G2820_R_GR_HEALTHY_NONLINEAR_DIRAC_DERIVED = False

assert not G2820_R_GR_QUADRATIC_D_KINETIC_PRESENT
assert not G2820_R_GR_QUADRATIC_D_ALGEBRAIC_TERM_PRESENT
assert G2820_R_GR_ACCIDENTAL_LINEARIZED_NULL_DIRECTIONS == 9
assert G2820_R_GR_PERTURBATIVE_STRONG_COUPLING_OBSTRUCTION
assert not G2820_R_GR_HEALTHY_NONLINEAR_DIRAC_DERIVED

# 9 — Strong coupling en approchant \(q_0\to0^+\)

Le problème n'est pas limité au point exact GR.

## Mode STF

Sur un fond :

\[
q=q_0+\delta q,
\]

la cinétique contient :

\[
\frac{K_S}{2}q(\partial\psi)^2.
\]

La normalisation canonique :

\[
\psi_c=\sqrt{K_Sq_0}\,\psi
\]

donne l'interaction :

\[
\boxed{
\frac{\delta q}{2q_0}
(\partial\psi_c)^2.
}
\]

Le coefficient diverge quand :

\[
q_0\to0.
\]

## Route R

Avec :

\[
q_c=\sqrt{\eta_Q}\,\delta q,
\]

le couplage devient :

\[
\boxed{
\frac{1}{2q_0\sqrt{\eta_Q}}
q_c(\partial\psi_c)^2.
}
\]

Donc la force de couplage diverge comme :

\[
\boxed{q_0^{-1}}.
\]

## Route P

Le gap est auxiliaire. Si :

\[
U''(q_0)=M_q
\]

reste fini et non nul, l'élimination de \(\delta q\) produit schématiquement :

\[
\boxed{
\Delta L_{\rm eff}
\sim
\frac{1}{8U''(q_0)q_0^2}
\left[(\partial\psi_c)^2\right]^2.
}
\]

Ce coefficient diverge comme :

\[
\boxed{q_0^{-2}}
\]

si \(U''\) reste fini.

Ainsi **P et R présentent toutes deux une échelle de couplage qui s'effondre lorsque la branche gappée approche GR**, sauf compensation non dérivée des coefficients.

In [10]:
q0,eta,Upp = sp.symbols(
    "q0 eta_Q U_pp",
    positive=True, real=True
)

g_R = sp.factor(1/(2*q0*sp.sqrt(eta)))
g_P4 = sp.factor(1/(8*Upp*q0**2))

lim_R = sp.limit(g_R,q0,0,dir="+")
lim_P = sp.limit(g_P4,q0,0,dir="+")

G2820_R_GAPPED_TO_GR_COUPLING_DIVERGES = (lim_R == sp.oo)
G2820_P_GAPPED_TO_GR_EFFECTIVE_COUPLING_DIVERGES_IF_UPP_FINITE = (lim_P == sp.oo)

assert G2820_R_GAPPED_TO_GR_COUPLING_DIVERGES
assert G2820_P_GAPPED_TO_GR_EFFECTIVE_COUPLING_DIVERGES_IF_UPP_FINITE

print("R cubic normalized coupling ~",g_R,"->",lim_R)
print("P induced quartic coupling ~",g_P4,"->",lim_P)

R cubic normalized coupling ~ 1/(2*sqrt(eta_Q)*q0) -> oo
P induced quartic coupling ~ 1/(8*U_pp*q0**2) -> oo


# 10 — Le tilt est encore plus sensible à la fermeture du gap

Le tilt possède :

\[
K_{\rm tilt}\propto q_0^2.
\]

Sa variable canonique se comporte comme :

\[
u_c\propto q_0u.
\]

Une fluctuation de gap dans :

\[
q^2(\partial u)^2
\]

produit après normalisation un couplage relatif :

\[
\boxed{
\sim\frac{\delta q}{q_0}
(\partial u_c)^2.
}
\]

La même divergence \(q_0^{-1}\) apparaît avant même de considérer les autres interactions.

Donc la chute :

\[
K_{\rm tilt}\sim q_0^2
\]

est une source structurelle de strong coupling à la fermeture du gap.

In [11]:
G2820_TILT_KINETIC_VANISHES_AS_Q2 = True
G2820_TILT_NORMALIZED_GAP_COUPLING_DIVERGES_AS_INV_Q = True

assert G2820_TILT_KINETIC_VANISHES_AS_Q2
assert G2820_TILT_NORMALIZED_GAP_COUPLING_DIVERGES_AS_INV_Q

# 11 — Le premier discriminant physique P/R

Les deux branches ne sont plus seulement différentes par leur comptage de DOF.

Sur le critère :

\[
\boxed{
\text{existence d'une branche GR quadratique contrôlable}
}
\]

elles diffèrent.

### P

Peut disposer d'un opérateur quadratique **algébrique** :

\[
U'(0)Q_D
\]

si :

\[
U'(0)\neq0.
\]

Cela peut éliminer conditionnellement les neuf composantes traceless comme paires seconde classe et laisser deux DOF GR.

### R

Dans sa définition actuelle :

\[
U=0,
\]

et tous les termes D-native réguliers commencent à l'ordre quartique autour de \(A=0\).

R possède donc une obstruction perturbative plus forte à GR.

Cela constitue :

\[
\boxed{
\textbf{un avantage structurel conditionnel de P sur R}
}
\]

si une branche GR perturbativement contrôlée est imposée comme critère physique.

Mais ce n'est pas encore une sélection fondamentale de P, parce que :

- \(U(q)\) n'est pas dérivé par GVH ;
- \(U'(0)\neq0\) n'est pas sélectionné ;
- la transition gappée \(q_0\to0\) de P est elle-même strong-coupled pour \(U''\) fini ;
- le Dirac non linéaire exact de la branche GR-P reste à faire.

In [12]:
selection = pd.DataFrame([
    {
        "criterion":"gapped local ghost-free domain",
        "GVH-P":"YES if KS>0, beta>0",
        "GVH-R":"YES if KS>0, beta>0, eta_Q>0",
        "discriminates":"NO",
    },
    {
        "criterion":"isolated constant-background metric cone",
        "GVH-P":"YES",
        "GVH-R":"YES incl. radial gap",
        "discriminates":"NO",
    },
    {
        "criterion":"global anisotropic ghost freedom",
        "GVH-P":"NOT PROVEN",
        "GVH-R":"NOT PROVEN",
        "discriminates":"NO",
    },
    {
        "criterion":"exact GR derivative quadratic D propagator",
        "GVH-P":"NO",
        "GVH-R":"NO",
        "discriminates":"NO",
    },
    {
        "criterion":"possible GR algebraic quadratic control",
        "GVH-P":"YES if U'(0)!=0",
        "GVH-R":"NO in current Route R",
        "discriminates":"CONDITIONAL P ADVANTAGE",
    },
    {
        "criterion":"gapped-to-GR strong coupling",
        "GVH-P":"YES for finite U''",
        "GVH-R":"YES",
        "discriminates":"BOTH OBSTRUCTED",
    },
    {
        "criterion":"unique physical selection",
        "GVH-P":"NOT YET",
        "GVH-R":"NOT YET",
        "discriminates":"NO FINAL SELECTION",
    },
])

G2820_PHYSICAL_SELECTION_TABLE_MATERIALIZED = True
G2820_P_CONDITIONAL_GR_REGULARITY_ADVANTAGE = True
G2820_UNIQUE_P_OR_R_SELECTION_DERIVED = False

assert G2820_PHYSICAL_SELECTION_TABLE_MATERIALIZED
assert G2820_P_CONDITIONAL_GR_REGULARITY_ADVANTAGE
assert not G2820_UNIQUE_P_OR_R_SELECTION_DERIVED

print(selection.to_string(index=False))

                                 criterion               GVH-P                        GVH-R           discriminates
            gapped local ghost-free domain YES if KS>0, beta>0 YES if KS>0, beta>0, eta_Q>0                      NO
  isolated constant-background metric cone                 YES         YES incl. radial gap                      NO
          global anisotropic ghost freedom          NOT PROVEN                   NOT PROVEN                      NO
exact GR derivative quadratic D propagator                  NO                           NO                      NO
   possible GR algebraic quadratic control     YES if U'(0)!=0        NO in current Route R CONDITIONAL P ADVANTAGE
              gapped-to-GR strong coupling  YES for finite U''                          YES         BOTH OBSTRUCTED
                 unique physical selection             NOT YET                      NOT YET      NO FINAL SELECTION


# 12 — Matching / dimensions / rang / SI ledger

`.28.20` produit des discriminants canoniques et de stabilité, pas une nouvelle calibration.

Les coefficients :

\[
K_S,\quad
\kappa_{\rm eff},\quad
\eta_Q,\quad
U'(0),\quad
U''(q_*)
\]

restent non identifiés en unités SI par un principe GVH.

Le fait qu'un mode isolé ait :

\[
\omega^2=k^2
\]

ne détermine pas son amplitude de normalisation cinétique.

Le rang SI universel reste donc :

\[
\boxed{0}.
\]

In [13]:
matching_ledger = pd.DataFrame([
    {
        "benchmark":"gapped P ghost domain",
        "GVH_quantity":"physical kinetic eigenvalues",
        "reference_quantity":"positive reduced Hessian",
        "matching_equation":"KS q>0; (16/9) beta q^2>0",
        "dimensions":"kinetic",
        "status":"LOCAL_PASS",
        "rank_increment":0,
        "SI_identifiability":"none",
        "degeneracies":"q=0,beta=0,anisotropic rank surfaces",
    },
    {
        "benchmark":"gapped R ghost domain",
        "GVH_quantity":"physical kinetic eigenvalues",
        "reference_quantity":"positive reduced Hessian",
        "matching_equation":"P conditions + eta_Q>0",
        "dimensions":"kinetic",
        "status":"LOCAL_PASS",
        "rank_increment":0,
        "SI_identifiability":"eta_Q not calibrated",
        "degeneracies":"q=0,eta_Q=0,anisotropic rank surfaces",
    },
    {
        "benchmark":"isolated characteristics",
        "GVH_quantity":"principal symbol",
        "reference_quantity":"metric cone",
        "matching_equation":"K(-omega^2+k^2)=0",
        "dimensions":"principal",
        "status":"LOCAL_CONSTANT_BACKGROUND_PASS",
        "rank_increment":0,
        "SI_identifiability":"none",
        "degeneracies":"full coupled anisotropic matrix open",
    },
    {
        "benchmark":"GR derivative quadratic action",
        "GVH_quantity":"S_D^(2)",
        "reference_quantity":"A=epsilon B",
        "matching_equation":"O_shape,O_angle,(dQ)^2 = O(epsilon^4)",
        "dimensions":"action",
        "status":"ZERO_QUADRATIC_DERIVATIVE_ACTION",
        "rank_increment":0,
        "SI_identifiability":"N/A",
        "degeneracies":"rank bifurcation",
    },
    {
        "benchmark":"P exact-GR algebraic escape",
        "GVH_quantity":"U'(0) Q_D",
        "reference_quantity":"9 traceless algebraic pairs",
        "matching_equation":"U'(0)!=0 => conditional S=18, N_DOF=2",
        "dimensions":"potential",
        "status":"CONDITIONAL_QUADRATIC_DIRAC_MODEL",
        "rank_increment":0,
        "SI_identifiability":"U'(0) not selected",
        "degeneracies":"U'(0)=0; nonlinear closure open",
    },
    {
        "benchmark":"R exact-GR perturbation",
        "GVH_quantity":"D quadratic action",
        "reference_quantity":"current Route R",
        "matching_equation":"kinetic=0 and algebraic quadratic=0",
        "dimensions":"action",
        "status":"STRONG_COUPLING_OBSTRUCTION",
        "rank_increment":0,
        "SI_identifiability":"N/A",
        "degeneracies":"could require new completion beyond current R",
    },
    {
        "benchmark":"P gapped-to-GR coupling",
        "GVH_quantity":"induced canonical quartic",
        "reference_quantity":"q0->0",
        "matching_equation":"~1/[8 U'' q0^2]",
        "dimensions":"interaction",
        "status":"DIVERGES_IF_U''_FINITE",
        "rank_increment":0,
        "SI_identifiability":"no strong-coupling scale calibrated",
        "degeneracies":"non-derived coefficient scaling could modify",
    },
    {
        "benchmark":"R gapped-to-GR coupling",
        "GVH_quantity":"canonical cubic",
        "reference_quantity":"q0->0",
        "matching_equation":"~1/[2 q0 sqrt(eta_Q)]",
        "dimensions":"interaction",
        "status":"DIVERGES",
        "rank_increment":0,
        "SI_identifiability":"no strong-coupling scale calibrated",
        "degeneracies":"fundamental-variable chart singular",
    },
])

G2820_MATCHING_LEDGER_MATERIALIZED = len(matching_ledger)==8
G2820_FORMAL_SOURCE_MATCHED_SCALE_RANK = 2
G2820_UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK = 0
G2820_ABSOLUTE_UNIVERSAL_SI_SCALE_DERIVED = False

assert G2820_MATCHING_LEDGER_MATERIALIZED
assert not G2820_ABSOLUTE_UNIVERSAL_SI_SCALE_DERIVED

print(matching_ledger.to_string(index=False))

                     benchmark                 GVH_quantity          reference_quantity                     matching_equation  dimensions                            status  rank_increment                  SI_identifiability                                  degeneracies
         gapped P ghost domain physical kinetic eigenvalues    positive reduced Hessian             KS q>0; (16/9) beta q^2>0     kinetic                        LOCAL_PASS               0                                none          q=0,beta=0,anisotropic rank surfaces
         gapped R ghost domain physical kinetic eigenvalues    positive reduced Hessian                P conditions + eta_Q>0     kinetic                        LOCAL_PASS               0                eta_Q not calibrated         q=0,eta_Q=0,anisotropic rank surfaces
      isolated characteristics             principal symbol                 metric cone                     K(-omega^2+k^2)=0   principal    LOCAL_CONSTANT_BACKGROUND_PASS               0   

# 13 — Verdict scientifique `.28.20`

## Ghosts sur branche gappée

\[
\boxed{
\textbf{LOCAL PASS}
}
\]

pour P si :

\[
K_S>0,\qquad\beta>0,
\]

et pour R si :

\[
K_S>0,\qquad\beta>0,\qquad\eta_Q>0.
\]

Mais :

\[
\boxed{
\textbf{GLOBAL GHOST-FREEDOM = NOT PROVEN}.
}
\]

## Caractéristiques

Sur fond constant/isotrope et pour les blocs physiques isolés :

\[
\boxed{
\omega^2=k^2.
}
\]

Mais :

\[
\boxed{
\textbf{FULL COUPLED CHARACTERISTIC MATRIX = OPEN}.
}
\]

## Branche GR

Tous les opérateurs dérivatifs polynomialement réguliers actuels du secteur \(D\) commencent à :

\[
\boxed{O(A^4)}.
\]

Donc :

\[
\boxed{
S_{D,\rm derivative}^{(2)}[A=0]=0.
}
\]

### P

Si :

\[
U'(0)\neq0,
\]

P possède conditionnellement un opérateur algébrique quadratique capable d'éliminer les neuf composantes traceless et de laisser :

\[
\boxed{N_{\rm DOF}^{GR,P}=2}
\]

dans le modèle Dirac quadratique.

### R

Dans sa forme actuelle, R ne possède ni cinétique ni potentiel quadratique \(D\) à GR.

Cela produit une dégénérescence linéarisée accidentelle et constitue :

\[
\boxed{
\textbf{une obstruction de strong coupling à la branche GR pour R}.
}
\]

## Approche gappée vers GR

Pour les deux branches, les normalisations canoniques deviennent singulières quand :

\[
q_0\to0^+.
\]

R :

\[
\boxed{
g_{\rm eff}^{R}\sim q_0^{-1}.
}
\]

P avec \(U''\) fini :

\[
\boxed{
g_{\rm eff}^{P}\sim q_0^{-2}
}
\]

pour l'interaction quartique induite après élimination du gap.

Donc la branche GR n'est pas obtenue comme une limite perturbative lisse des branches gappées actuelles.

### Sélection P/R

`.28.20` obtient pour la première fois un **avantage physique conditionnel de P** :

\[
\boxed{
P\text{ peut posséder une branche GR algébriquement contrôlée si }U'(0)\neq0,
}
\]

alors que :

\[
\boxed{
R\text{ actuelle est perturbativement fortement couplée à }A=0.
}
\]

Mais P n'est pas encore sélectionnée comme théorie unique, parce que \(U(q)\) n'est pas dérivé et que la transition gappée-vers-GR reste elle-même strong-coupled.

Le verdict correct est donc :

\[
\boxed{
\textbf{R DISFAVORED AT EXACT GR IN CURRENT COMPLETION;}
}
\]

\[
\boxed{
\textbf{P CONDITIONALLY FAVORED, NOT FUNDAMENTALLY SELECTED.}
}
\]

In [14]:
LEVEL1 = "GENERIC_GAPPED_P_AND_R_HAVE_NONEMPTY_LOCAL_GHOST_FREE_DOMAINS_AND_ISOLATED_CONSTANT_BACKGROUND_METRIC_CONES"
LEVEL2 = "ALL_CURRENT_POLYNOMIAL_D_DERIVATIVE_OPERATORS_BEGIN_AT_QUARTIC_ORDER_AROUND_A_ZERO_SO_THE_GR_BRANCH_HAS_NO_D_QUADRATIC_DERIVATIVE_PROPAGATOR"
LEVEL3 = "CURRENT_R_HAS_A_GR_STRONG_COUPLING_OBSTRUCTION_WHILE_P_HAS_A_CONDITIONAL_ALGEBRAIC_GR_ESCAPE_IF_UPRIME_AT_ZERO_IS_NONZERO"
LEVEL4 = "BOTH_GAPPED_BRANCHES_DEVELOP_DIVERGENT_CANONICALLY_NORMALIZED_INTERACTIONS_AS_Q0_APPROACHES_ZERO_SO_NO_SMOOTH_GAPPED_TO_GR_PERTURBATIVE_LIMIT_IS_ESTABLISHED"

G2820_GAPPED_LOCAL_GHOST_GATE_P_PASS = True
G2820_GAPPED_LOCAL_GHOST_GATE_R_PASS = True
G2820_GAPPED_LOCAL_CHARACTERISTIC_GATE_PASS = True

G2820_GR_D_QUADRATIC_DERIVATIVE_PROPAGATOR_PRESENT = False
G2820_R_GR_STRONG_COUPLING_GATE_FAIL = True
G2820_P_GR_CONDITIONAL_ALGEBRAIC_ESCAPE = True

G2820_SMOOTH_GAPPED_TO_GR_LIMIT_P_PROVEN = False
G2820_SMOOTH_GAPPED_TO_GR_LIMIT_R_PROVEN = False

G2820_P_PHYSICALLY_SELECTED_OVER_R_CONDITIONALLY = True
G2820_P_FUNDAMENTALLY_SELECTED_BY_GVH = False
G2820_R_FUNDAMENTALLY_EXCLUDED_BY_FULL_NONLINEAR_THEORY = False

G2820_FULL_COUPLED_CHARACTERISTICS_CLOSED = False
G2820_FULL_GLOBAL_GHOST_FREEDOM_PROVEN = False
G2820_FULL_NONLINEAR_GR_BRANCH_DIRAC_CLOSED = False
G2820_FULL_PURE_GVH_CLASSICAL_ACTION_CLOSED = False
G2820_REAL_DATA_READY = False
G2820_NEW_GVH_PHYSICS_VALIDATED = False

G2820_LOCAL_AUDIT_PASS = all([
    G2820_UPSTREAM_GATE,
    G2820_P_GAPPED_KINETIC_POSITIVE,
    G2820_R_GAPPED_KINETIC_POSITIVE,
    G2820_MIXED_HESSIAN_CONGRUENCE_PASS,
    G2820_ANISOTROPIC_HEALTHY_DOMAIN_NONEMPTY,
    G2820_GAPPED_ISOTROPIC_ISOLATED_MODE_METRIC_CONE,
    G2820_GR_D_DERIVATIVE_QUADRATIC_ACTION_ZERO,
    G2820_NORMALIZED_DCC_PATH_DEPENDENT_AT_ORIGIN,
    G2820_P_GR_ALGEBRAIC_REGULARIZATION_AVAILABLE_IF_UPRIME0_NONZERO,
    G2820_P_GR_CONDITIONAL_DOF == 2,
    G2820_R_GR_PERTURBATIVE_STRONG_COUPLING_OBSTRUCTION,
    G2820_R_GAPPED_TO_GR_COUPLING_DIVERGES,
    G2820_P_GAPPED_TO_GR_EFFECTIVE_COUPLING_DIVERGES_IF_UPP_FINITE,
    G2820_TILT_NORMALIZED_GAP_COUPLING_DIVERGES_AS_INV_Q,
    G2820_P_CONDITIONAL_GR_REGULARITY_ADVANTAGE,
    not G2820_UNIQUE_P_OR_R_SELECTION_DERIVED,
    not G2820_FULL_COUPLED_CHARACTERISTICS_CLOSED,
    not G2820_FULL_GLOBAL_GHOST_FREEDOM_PROVEN,
    not G2820_FULL_NONLINEAR_GR_BRANCH_DIRAC_CLOSED,
    not G2820_FULL_PURE_GVH_CLASSICAL_ACTION_CLOSED,
    not G2820_REAL_DATA_READY,
])

G2820_OBSTRUCTIONS = [
    "FULL_COUPLED_ANISOTROPIC_CHARACTERISTIC_MATRIX_NOT_DERIVED",
    "GLOBAL_GHOST_FREEDOM_NOT_PROVEN_OUTSIDE_OPEN_GAPPED_HEALTHY_DOMAIN",
    "P_GR_TWO_DOF_RESULT_IS_CONDITIONAL_ON_U_PRIME_AT_ZERO_NONZERO_AND_QUADRATIC_ALGEBRAIC_NONDEGENERACY",
    "P_POTENTIAL_U_IS_NOT_SELECTED_BY_A_FUNDAMENTAL_GVH_PRINCIPLE",
    "R_CURRENT_COMPLETION_HAS_NO_QUADRATIC_D_ACTION_AT_EXACT_GR_AND_IS_PERTURBATIVELY_STRONG_COUPLED",
    "P_AND_R_GAPPED_BRANCHES_BOTH_HAVE_CANONICAL_INTERACTION_COUPLINGS_DIVERGING_AS_Q0_TO_ZERO",
    "FULL_NONLINEAR_FUNDAMENTAL_A_DIRAC_ANALYSIS_AT_EXACT_GR_REMAINS_OPEN",
    "FULL_HISTORICAL_DCC_GLOBAL_OPERATOR_EQUIVALENCE_REMAINS_OPEN",
]

G2820_NEXT_AUTHORIZED = (
    "0.3.2.7.3.7.3.3.28.20.1_"
    "Pure_GVH_P_GR_Branch_Fundamental_A_Dirac_U_Principle_and_Strong_Coupling_Resolution_Audit"
)

assert G2820_LOCAL_AUDIT_PASS

print("LEVEL1 =",LEVEL1)
print("LEVEL2 =",LEVEL2)
print("LEVEL3 =",LEVEL3)
print("LEVEL4 =",LEVEL4)
print("G2820_LOCAL_AUDIT_PASS =",G2820_LOCAL_AUDIT_PASS)
print("G2820_P_PHYSICALLY_SELECTED_OVER_R_CONDITIONALLY =",
      G2820_P_PHYSICALLY_SELECTED_OVER_R_CONDITIONALLY)
print("G2820_P_FUNDAMENTALLY_SELECTED_BY_GVH =",
      G2820_P_FUNDAMENTALLY_SELECTED_BY_GVH)
print("G2820_R_GR_STRONG_COUPLING_GATE_FAIL =",
      G2820_R_GR_STRONG_COUPLING_GATE_FAIL)
print("G2820_NEXT_AUTHORIZED =",G2820_NEXT_AUTHORIZED)

LEVEL1 = GENERIC_GAPPED_P_AND_R_HAVE_NONEMPTY_LOCAL_GHOST_FREE_DOMAINS_AND_ISOLATED_CONSTANT_BACKGROUND_METRIC_CONES
LEVEL2 = ALL_CURRENT_POLYNOMIAL_D_DERIVATIVE_OPERATORS_BEGIN_AT_QUARTIC_ORDER_AROUND_A_ZERO_SO_THE_GR_BRANCH_HAS_NO_D_QUADRATIC_DERIVATIVE_PROPAGATOR
LEVEL3 = CURRENT_R_HAS_A_GR_STRONG_COUPLING_OBSTRUCTION_WHILE_P_HAS_A_CONDITIONAL_ALGEBRAIC_GR_ESCAPE_IF_UPRIME_AT_ZERO_IS_NONZERO
LEVEL4 = BOTH_GAPPED_BRANCHES_DEVELOP_DIVERGENT_CANONICALLY_NORMALIZED_INTERACTIONS_AS_Q0_APPROACHES_ZERO_SO_NO_SMOOTH_GAPPED_TO_GR_PERTURBATIVE_LIMIT_IS_ESTABLISHED
G2820_LOCAL_AUDIT_PASS = True
G2820_P_PHYSICALLY_SELECTED_OVER_R_CONDITIONALLY = True
G2820_P_FUNDAMENTALLY_SELECTED_BY_GVH = False
G2820_R_GR_STRONG_COUPLING_GATE_FAIL = True
G2820_NEXT_AUTHORIZED = 0.3.2.7.3.7.3.3.28.20.1_Pure_GVH_P_GR_Branch_Fundamental_A_Dirac_U_Principle_and_Strong_Coupling_Resolution_Audit


In [15]:
artifact = {
    "notebook":(
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.20_"
        "Pure_GVH_PR_Ghost_Characteristics_GR_Branch_and_Strong_Coupling_Audit_FAST"
    ),
    "branch":"PURE_GVH_PR_COMPARATIVE",
    "upstream":{
        "P":P,
        "R":R,
    },
    "levels":{
        "LEVEL1":LEVEL1,
        "LEVEL2":LEVEL2,
        "LEVEL3":LEVEL3,
        "LEVEL4":LEVEL4,
    },
    "ghosts":{
        "P_gapped_local_pass":G2820_GAPPED_LOCAL_GHOST_GATE_P_PASS,
        "R_gapped_local_pass":G2820_GAPPED_LOCAL_GHOST_GATE_R_PASS,
        "global_proof":G2820_FULL_GLOBAL_GHOST_FREEDOM_PROVEN,
    },
    "characteristics":{
        "isolated_constant_background_metric_cone":
            G2820_GAPPED_ISOTROPIC_ISOLATED_MODE_METRIC_CONE,
        "full_coupled_closed":
            G2820_FULL_COUPLED_CHARACTERISTICS_CLOSED,
    },
    "GR":{
        "D_derivative_quadratic_action_zero":
            G2820_GR_D_DERIVATIVE_QUADRATIC_ACTION_ZERO,
        "P_conditional_algebraic_DOF":
            G2820_P_GR_CONDITIONAL_DOF,
        "P_requires_Uprime0_nonzero":True,
        "R_strong_coupling_obstruction":
            G2820_R_GR_PERTURBATIVE_STRONG_COUPLING_OBSTRUCTION,
        "full_nonlinear_Dirac_closed":
            G2820_FULL_NONLINEAR_GR_BRANCH_DIRAC_CLOSED,
    },
    "strong_coupling":{
        "R_gapped_to_GR_diverges":
            G2820_R_GAPPED_TO_GR_COUPLING_DIVERGES,
        "P_gapped_to_GR_diverges_if_Upp_finite":
            G2820_P_GAPPED_TO_GR_EFFECTIVE_COUPLING_DIVERGES_IF_UPP_FINITE,
        "tilt_gap_coupling_diverges":
            G2820_TILT_NORMALIZED_GAP_COUPLING_DIVERGES_AS_INV_Q,
    },
    "selection":{
        "P_conditional_advantage":
            G2820_P_CONDITIONAL_GR_REGULARITY_ADVANTAGE,
        "P_fundamentally_selected":
            G2820_P_FUNDAMENTALLY_SELECTED_BY_GVH,
        "R_fundamentally_excluded":
            G2820_R_FUNDAMENTALLY_EXCLUDED_BY_FULL_NONLINEAR_THEORY,
    },
    "matching":matching_ledger.to_dict(orient="records"),
    "locks":{
        "full_pure_GVH_classical_action_closed":
            G2820_FULL_PURE_GVH_CLASSICAL_ACTION_CLOSED,
        "real_data_ready":
            G2820_REAL_DATA_READY,
    },
    "verdict":{
        "local_audit_pass":G2820_LOCAL_AUDIT_PASS,
        "obstructions":G2820_OBSTRUCTIONS,
        "next_authorized":G2820_NEXT_AUTHORIZED,
    },
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.20_"
    "Pure_GVH_PR_Ghost_Characteristics_GR_Branch_and_Strong_Coupling_Audit_FAST.json"
)
artifact_path.write_text(
    json.dumps(artifact,indent=2,ensure_ascii=False),
    encoding="utf-8",
)

print("G2820_LOCAL_AUDIT_PASS =",G2820_LOCAL_AUDIT_PASS)
print("artifact =",artifact_path)

G2820_LOCAL_AUDIT_PASS = True
artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.28.20_Pure_GVH_PR_Ghost_Characteristics_GR_Branch_and_Strong_Coupling_Audit_FAST.json
